# Example 1: Weighted PINN baseline

Run from the repository root. The notebook trains the baseline at six weights and saves each run under `outputs/weighted_pinn/`.

### Optimization Problem (P) - Minimum Potential Energy Formulation

$$
\begin{aligned}
(\mathscr{P}) \quad \min_{v \in U_{ad}} \quad & J(y, v) = \frac{1}{2} \|y - z_d\|_{L^2(\Omega)}^2 + \frac{V}{2} \|v\|_{L^2(\Omega)}^2 \\
\text{subject to} \quad & y = \argmin_{w \in K} \mathscr{E}(w; v) \\
& K = \{ w \in H_0^1(\Omega) \mid w \geq 0 \text{ a.e. in } \Omega \}
\end{aligned}
$$

#### Potential Energy Functional
The energy functional $\mathscr{E}(w; v)$ is given by:

$$
\mathscr{E}(w; v) = \frac{1}{2}a(w,w) - \langle f + v, w \rangle
$$

where:
- $a(w,w) = \int_\Omega \nabla w \cdot \nabla w \, dx$ (Dirichlet energy)
- $\langle f + v, w \rangle = \int_\Omega (f + v)w \, dx$ (work done by external forces)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from   torch.autograd import grad
from utils import *
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader,TensorDataset
from scipy.stats.qmc import LatinHypercube
from functools import partial
from torch.optim.lr_scheduler import LambdaLR
import os
import time 
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from datetime import datetime
# Define ROOT directory for saving results
ROOT = Path('outputs/weighted_pinn')

In [ ]:
def J_u(source,net_y,net_v,x):
    """Upper-level objective evaluated by sample means."""
    yd = source.compute_yd
    nu = source.nu
    term1 = (1/2)*nn.MSELoss()(net_y(x),yd(x))
    term2 = (nu/2)*nn.MSELoss()(net_v(x),torch.zeros_like(net_v(x)).to(device))

    return term1 +term2


def J_l(source, net_y, net_v, x):
    """Lower-level energy for the given control."""
    f = source.compute_f
    x = x.clone().detach().requires_grad_(True)
    y_pred = net_y(x)
    v_pred = net_v(x)
    f_val = f(x)


    grads = grad(y_pred.sum(), x, create_graph=True)[0]
    y_x1 = grads[:, 0:1]
    y_x2 = grads[:, 1:2]



    a_val = y_x1**2 + y_x2**2
    linear_term = (f_val+v_pred)*y_pred

    return torch.mean(1/2*a_val-linear_term)

In [ ]:
def rel_l2_error_yv(net_y, net_v, source, device='cpu'):
    """Relative L2 state and control errors on the original evaluation grids."""

    x = torch.linspace(0, 1, 200, device=device)
    X, Y = torch.meshgrid(x, x, indexing='ij')

    xy = torch.cat((X.reshape(-1,1), Y.reshape(-1,1)), dim=1)
    X, Y = torch.meshgrid(x[1:-1], x[1:-1], indexing='ij')
    xy_interior = torch.cat((X.reshape(-1,1), Y.reshape(-1,1)), dim=1)

    with torch.no_grad():
        y_star = source.compute_y_star(xy)
        v_star = source.compute_u_star(xy_interior)
    

    y_app = net_y(xy)  
    v_app = net_v(xy_interior)  


    num_y = torch.sum((y_app - y_star) ** 2)
    den_y = torch.sum(y_star ** 2)
    num_v = torch.sum((v_app - v_star) ** 2)
    den_v = torch.sum(v_star ** 2)


    rel_l2_y = torch.sqrt(num_y / (den_y + 1e-30))
    rel_l2_v = torch.sqrt(num_v / (den_v + 1e-30))


    print(f"Relative L2 error for y: {rel_l2_y.item():.4e}")
    print(f"Relative L2 error for v: {rel_l2_v.item():.4e}")

    return rel_l2_y, rel_l2_v

# Random interior sampling

In [ ]:
class RandomDataLoader:
    """Random interior batches."""
    def __init__(self, batch_size=64, steps_per_epoch=4, dims=2, device='cpu',
                ):
        """Initialize the component with the supplied settings."""
        self.steps_per_epoch = steps_per_epoch
        self.dims = dims
        self.device = device
        self.current_step = 0
        

        self.interior_points = max(1, batch_size)
        

        self.batch_size = self.interior_points
        
    def __iter__(self):
        """Start a new batch iterator."""
        self.current_step = 0
        return self
    
    def __next__(self):
        """Generate the next batch of random interior points."""
        if self.current_step < self.steps_per_epoch:
            self.current_step += 1
            

            interior_points = torch.rand(self.interior_points, self.dims, device=self.device)

            return (interior_points,)
        else:

            self.current_step = 0
            raise StopIteration
    
    def __len__(self):
        """Return the number of batches per epoch."""
        return self.steps_per_epoch


In [ ]:
def get_xi(source, net_y, net_v, x):
    """Compute xi = -Delta y - v - f."""
    x = x.clone().detach().requires_grad_(True)
    x1 = x[:, 0:1]
    x2 = x[:, 1:2]
    x = torch.cat([x1, x2], 1)
    y_pred  = net_y(x)
    v_pred  = net_v(x)


    f_val = source.compute_f(x)
    


    y_x1 = grad(y_pred.sum(), x1, create_graph=True)[0]
    y_x2 = grad(y_pred.sum(), x2, create_graph=True)[0]

    d2y_dx1 = grad(y_x1.sum(), x1, create_graph=True)[0]
    d2y_dx2 = grad(y_x2.sum(), x2, create_graph=True)[0]
    neg_lap_y = -(d2y_dx1 + d2y_dx2)

    xi = neg_lap_y - v_pred - f_val
    return xi

def ortho_loss(source,net_y, net_v, x):
    """Orthogonality penalty."""
    y_pred  = net_y(x)
    xi = get_xi(source, net_y, net_v, x)

    return torch.mean((y_pred*xi)**2)

def non_neg_loss(source,net_y,net_v, x):
    """Nonnegativity penalty for xi."""
    xi = get_xi(source, net_y, net_v, x)
    return torch.mean((torch.relu(-xi))**2)

In [ ]:
def export_to_matlab(net_y, net_u, source, grid_size=52, device='cuda', save_path=None):
    """Export sampled state, control, and multiplier arrays."""
    import numpy as np
    import torch
    from torch.autograd import grad
    import os
    import scipy.io as sio
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D
    

    net_y.eval()
    net_u.eval()
    

    x = torch.linspace(0, 1, grid_size, device=device)
    X, Y = torch.meshgrid(x, x, indexing='ij')
    

    xy_points = torch.cat((X.reshape(-1, 1), Y.reshape(-1, 1)), dim=1).requires_grad_(True)
    

    xx = X.cpu().numpy()
    yy = Y.cpu().numpy()
    

    with torch.no_grad():

        y_values = net_y(xy_points).reshape(-1, 1)
        

        u_values = net_u(xy_points).reshape(-1, 1)
    

    xi_values = torch.relu(get_xi(source, net_y, net_u, xy_points))
    

    y_grid = y_values.reshape(grid_size, grid_size).cpu().numpy()
    u_grid = u_values.reshape(grid_size, grid_size).cpu().numpy()
    xi_grid = xi_values.reshape(grid_size, grid_size).detach().cpu().numpy() 
    

    matlab_data = {
        'X': xx,
        'Y': yy,
        'y_values': y_grid,
        'u_values': u_grid,
        'xi_values': xi_grid,
    }
    

    if save_path:

        save_dir = os.path.dirname(save_path)
        if save_dir and not os.path.exists(save_dir):
            os.makedirs(save_dir)
        

        sio.savemat(f"{save_path}.mat", matlab_data)
        print(f"Data saved to {save_path}.mat")
        

        np.savetxt(f"{save_path}_y.csv", y_grid, delimiter=',')
        np.savetxt(f"{save_path}_u.csv", u_grid, delimiter=',')
        np.savetxt(f"{save_path}_xi.csv", xi_grid, delimiter=',')
        print(f"CSV files saved to {save_path}_*.csv")
    

    fig = plt.figure(figsize=(15, 5))
    

    ax1 = fig.add_subplot(131, projection='3d')
    surf1 = ax1.plot_surface(xx, yy, y_grid, cmap='viridis')
    ax1.set_title('State y')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    

    ax2 = fig.add_subplot(132, projection='3d')
    surf2 = ax2.plot_surface(xx, yy, u_grid, cmap='viridis')
    ax2.set_title('Control u')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    

    ax3 = fig.add_subplot(133, projection='3d')
    surf3 = ax3.plot_surface(xx, yy, xi_grid, cmap='viridis')
    ax3.set_title('Multiplier xi')
    ax3.set_xlabel('x')
    ax3.set_ylabel('y')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(f"{save_path}_visualization.png", dpi=300, bbox_inches='tight')
    
    
    return matlab_data

In [ ]:
class Trainer:
    def __init__(
        self,
        v: nn.Module,
        y: nn.Module,
        source: Source,
        train_loader: DataLoader,
        optimizer: torch.optim.Optimizer,
        scheduler: torch.optim.lr_scheduler.LambdaLR, 
        device: torch.device = torch.device("cpu"),
        num_epochs: int = 10000,
        weight: list = [1, 1],
        save: bool = False,
        w: float = 0.0
    ):    
        """Initialize the component with the supplied settings."""
    
        self.v = v.to(device)
        self.y = y.to(device)
        self.source = source
        self.train_loader = train_loader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.num_epochs = num_epochs
        self.weight = weight
        self.save = save
        self.record_u = torch.zeros(1).to(self.device)
        self.record_l = torch.zeros(1).to(self.device)
        self.record_ortho = torch.zeros(1).to(self.device)
        self.record_non_neg = torch.zeros(1).to(self.device)
        self.timestart = 0
        self.w = w
        
        # Define folder suffix based on parameters
        self.file_suffix = f"{datetime.now().strftime('%Y%m%d')}pinn_weight_{self.w}_constraint_False"
            
        # Create directory and CSV files for logging if save=True
        if self.save:
            os.makedirs(f'{ROOT}/{self.file_suffix}', exist_ok=True)
            # Create loss CSV file with header
            self.loss_csv_file = f'{ROOT}/{self.file_suffix}/loss.csv'
            with open(self.loss_csv_file, 'w') as f:
                f.write("epoch,time,J_u_loss,J_l_loss,ortho_loss,non_neg_loss\n")
            # Create error metrics CSV file with header
            self.norm_csv_file = f'{ROOT}/{self.file_suffix}/norm.csv'
            with open(self.norm_csv_file, 'w') as f:
                f.write("epoch,time,rel_l2_error_y,rel_l2_error_v\n")
            # Create forward energy log
            self.forward_loss_file = f'{ROOT}/{self.file_suffix}/forward_loss.csv'
            with open(self.forward_loss_file, 'w') as f:
                f.write("epoch,time,J_total_avg,lower_energy\n")

    def getloss_default(self, x):
        x = x.clone().detach().requires_grad_(True)
        return J_u(self.source, self.y, self.v, x), J_l(self.source, self.y, self.v, x)
    
    def train_one_epoch(self, epoch, timestart):
        self.v.train()
        self.y.train()
        total_loss = 0.0
        batch_count = 0
        for batch in self.train_loader:
            x = batch[0].requires_grad_().to(self.device)
            self.optimizer.zero_grad()
            J_u_loss, J_l_loss = self.getloss_default(x)


            loss = self.weight[0] * J_u_loss + self.weight[1] * J_l_loss
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
            batch_count += 1
        self.scheduler.step()
        avg_loss = total_loss / batch_count if batch_count > 0 else 0
        
        if (epoch-1) % 10 ==0:        

            grid_size = 50
            x1 = torch.linspace(0, 1, grid_size, device=self.device)
            x2 = torch.linspace(0, 1, grid_size, device=self.device)
            X1, X2 = torch.meshgrid(x1, x2, indexing='ij')
            grid_points = torch.stack([X1.flatten(), X2.flatten()], dim=1).requires_grad_(True)
            self.record_u = J_u(self.source, self.y, self.v, grid_points)
            self.record_l = J_l(self.source, self.y, self.v, grid_points)
            self.record_ortho = ortho_loss(self.source, self.y, self.v, grid_points)
            self.record_non_neg = non_neg_loss(self.source, self.y, self.v, grid_points)
            
            # Log loss values to CSV if save is enabled
            if self.save:
                with open(self.loss_csv_file, 'a') as f:
                    f.write(f"{epoch},{time.time()-timestart},{self.record_u.item()},{self.record_l.item()},{self.record_ortho.item()},{self.record_non_neg.item()}\n")
            # Log forward energy (lower) using compute_energy on 101 grid
            if self.save:
                lower_energy, _ = compute_energy(
                    self.source,
                    net_y=self.y,
                    net_u=self.v,
                    grid_size=101,
                    device=self.device,
                    requires_grad=False,
                )
                with open(self.forward_loss_file, 'a') as f:
                    f.write(f"{epoch},,{avg_loss},{lower_energy}\n")
            
            # Clear cache periodically
            if epoch % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()

        return self.record_u.item(), self.record_l.item(), self.record_ortho.item(), self.record_non_neg.item()
    
    def validate(self,epoch):
        self.v.eval()
        self.y.eval()
        plt.close('all')
        rel_l2_error_y, rel_l2_error_v = rel_l2_error_yv(self.y, self.v, self.source, device=self.device)
        
        # Log error metrics to CSV if save is enabled
        if self.save:
            with open(self.norm_csv_file, 'a') as f:
                f.write(f"{epoch},{time.time()-self.timestart},{rel_l2_error_y.item()},{rel_l2_error_v.item()}\n")
                
        with torch.no_grad():
            if self.save:
                yfile = f'{ROOT}/{self.file_suffix}/y_net_{epoch}.png'
                vfile = f'{ROOT}/{self.file_suffix}/v_net_{epoch}.png'
                xifile = f'{ROOT}/{self.file_suffix}/xi_net_{epoch}.png'
            else:
                yfile = None
                vfile = None
                xifile = None
                
        #     visual_y_3d(self.source, self.y, file_name=yfile)
        #     visual_v_3d(self.source, self.v, file_name=vfile)
            
        # # Visualize xi if constraint is enabled
        # if xifile is not None:
        #     visual_xi_3d(self.source, self.y, self.v, file_name=xifile)
        

    def fit(self):
        pbar = tqdm(range(1, self.num_epochs + 1), ascii=True, dynamic_ncols=True)
        self.timestart = time.time()
        for epoch in pbar:

            J_u_loss, J_l_loss, xi_ortho, xi_non_neg = self.train_one_epoch(epoch, self.timestart)

            pbar.set_postfix(
                J_u_loss=J_u_loss,
                J_l_loss=J_l_loss,
                xi_ortho=xi_ortho,
                xi_non_neg=xi_non_neg,
                time=time.time() - self.timestart,
            )
            if epoch % 2000 == 0 or epoch == 1:
                self.validate(epoch)


                output_dir = os.path.join(ROOT, self.file_suffix, "matlab_data", f"epoch_{epoch}")
                os.makedirs(output_dir, exist_ok=True)


                matlab_data = export_to_matlab(
                    net_y=self.y, 
                    net_u=self.v, 
                    source=self.source,
                    grid_size=101,
                    device=self.device,
                    save_path=os.path.join(output_dir, "solution_data")
                )

                print(f"Epoch {epoch}: data export complete")

# Device

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
source = Source5(device=device,scale=20)

# Hyperparameters

The first two loss weights are normalized by their sum.

In [ ]:
# hyperparameters
lr = 1e-3
warmup = 0
cycle_factor = 0.8
cycle_length = 1000
num_epochs = 20000
save = True  # enable data recording
batch_size = 512

# sweep weight[1] values (log10 = -1, -0.5, 0, 0.5, 1) plus 5.0 (~10^0.699)
weight_sweep = [10**-1, 10**-0.5, 10**0, 10**0.5, 10**1, 5.0]
print("weight[1] sweep:", weight_sweep)

In [ ]:
def hard(x1, x2):
    return x1 * (1 - x1) * x2 * (1 - x2)

In [ ]:
def build_components():
    """Recreate dataloader and fresh networks for each weight run."""
    dataloader = RandomDataLoader(
        batch_size=batch_size,
        steps_per_epoch=4,
        device=device,
    )
    y_net = net_y_res(source=source, hidden=16, layers=4, activation='Swish').to(device)
    v_net = net_v_res(source=source, hidden=16, layers=4, hard_=hard, activation='Swish').to(device)
    return dataloader, y_net, v_net


In [ ]:
for w1 in weight_sweep:
    print(f"\n=== Running sweep with weight[1] = {w1:.4g} ===")
    dataloader, y, v = build_components()
    current_weight = [1.0, w1]
    norm12 = current_weight[0] + current_weight[1]
    if norm12 > 0:
        current_weight[0] /= norm12
        current_weight[1] /= norm12
    optimizer = torch.optim.Adam(list(y.parameters()) + list(v.parameters()), lr=lr)
    scheduler = LambdaLR(optimizer, partial(lr_lambda, warmup=warmup, cycle_factor=cycle_factor, cycle_length=cycle_length))
    trainer = Trainer(v, y, source, dataloader, optimizer, scheduler, device, num_epochs=num_epochs, weight=current_weight, save=save, w = w1)
    trainer.fit()
    
    # Save models for this sweep value
    model_dir = os.path.join(ROOT, trainer.file_suffix, "models")
    os.makedirs(model_dir, exist_ok=True)
    torch.save(trainer.y.state_dict(), os.path.join(model_dir, "final_y.pth"))
    torch.save(trainer.v.state_dict(), os.path.join(model_dir, "final_u.pth"))